## Micrograd

In [212]:
import math

class Value:

    def __init__(self, data, _children = (), op = (), label = " "):
        self.data = data
        self.grad = 0
        self._backward = lambda: None 
        self._prev = set(_children)
        self._op = op
        self._label = label
    
    def __repr__(self):
        return f"Value(label={self._label}, data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        
        out._backward = _backward 
        return out

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward 
        return out

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * (other**-1)

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int and float powers are supported for now"
        out = Value(self.data ** other, (self,), f"**{other}")

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1)/(math.exp(2*n) + 1)
        out = Value(t, (self, ), label="tanh")

        def _backward():
            self.grad +=  (1 - t**2) * out.grad
        
        out._backward = _backward 
        return out

    def backprop(self):
        self._backward()
        print(f"label: {self._label}, grad: {self.grad}")
        for child in self._prev:
            child.backprop()

In [213]:
a = Value(3.0, label="a")
b = a + a
b.grad = 1
b.backprop()

label:  , grad: 1
label: a, grad: 2.0


In [214]:
a + 1
a * 2
2 * a
2 + a

Value(label= , data=5.0)

In [215]:
x1 = Value(2.0, label="x1")
x2 = Value(0.0, label="x2")

w1 = Value(-3.0, label="w1")
w2 = Value(1.0, label="w2")

b = Value(6.8813735870195432, label="b")

x1w1 = x1*w1
x1w1._label = "x1*w1"

x2w2 = x2*w2
x2w2._label = "x2*w2"

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2._label = "w1*w1 + x2*w2"

n = x1w1x2w2 + b
n._label = "n"

o = n.tanh()
o._label = "o"

In [216]:
o.grad = 1.0

In [217]:
o.backprop()

label: o, grad: 1.0
label: n, grad: 0.4999999999999999
label: b, grad: 0.4999999999999999
label: w1*w1 + x2*w2, grad: 0.4999999999999999
label: x2*w2, grad: 0.4999999999999999
label: w2, grad: 0.0
label: x2, grad: 0.4999999999999999
label: x1*w1, grad: 0.4999999999999999
label: x1, grad: -1.4999999999999996
label: w1, grad: 0.9999999999999998


In [218]:
import random

class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for i in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        assert len(x) == len(self.w), "number of inputs is not equal to the number of weights"
        act = sum([xi * self.w[i] for i, xi in enumerate(x)]) + self.b
        out = act.tanh()
        return out

In [219]:
x = [2.0, 3.0]
n = Neuron(len(x))
n(x)

Value(label=tanh, data=-0.9723284425708287)

In [220]:
class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out[0] if len(out) == 1 else out

In [221]:
x = [2.0, 3.0]
la = Layer(len(x), 3)
la(x)

[Value(label=tanh, data=0.9780429585500882),
 Value(label=tanh, data=-0.9148855697776663),
 Value(label=tanh, data=0.8923412044851583)]

In [222]:
class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        out = x
        for layer in self.layers:
            out = layer(out)
        return out

In [223]:
x = [2.0, 3.0, -1.0]
mlp = MLP(len(x), [4, 4, 1])
mlp(x)

Value(label=tanh, data=0.5491096041471355)

In [224]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0]

ypreds = [mlp(x) for x in xs]
ypreds

[Value(label=tanh, data=0.5491096041471355),
 Value(label=tanh, data=0.6228176229883742),
 Value(label=tanh, data=0.6996538868124366),
 Value(label=tanh, data=0.5832278885034872)]

In [225]:
loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
loss._label = "loss"
loss

Value(label=loss, data=5.899361514431874)

In [226]:
loss.grad = 1.0
loss.backprop()

label: loss, grad: 1.0
label:  , grad: 1.0
label:  , grad: 1.0
label:  , grad: 3.399307773624873
label:  , grad: 3.399307773624873
label: tanh, grad: 3.399307773624873
label:  , grad: 1.7352937206795127
label:  , grad: 1.7352937206795127
label:  , grad: 1.7352937206795127
label:  , grad: 1.7352937206795127
label:  , grad: 1.7352937206795127
label:  , grad: 1.7352937206795127
label:  , grad: 1.7352937206795127
label:  , grad: 0.3408736828587279
label: tanh, grad: 0.20587975828889604
label:  , grad: 0.19793547715135584
label:  , grad: 0.19793547715135584
label:  , grad: 0.19793547715135584
label:  , grad: 0.19793547715135584
label:  , grad: 0.19793547715135584
label:  , grad: 0.19793547715135584
label:  , grad: 0.19793547715135584
label:  , grad: 0.19793547715135584
label: tanh, grad: -0.0011726851551501413
label:  , grad: -0.00019806346167183935
label:  , grad: -0.00019806346167183935
label:  , grad: -0.00019806346167183935
label:  , grad: -0.00019806346167183935
label:  , grad: 0.00018